# Sub-Decision Prosodic Sensitivity — Progress Demo

**Question.** When Qwen2-Audio gets a contrastive-stress item wrong, does prosody still shift the sub-decision distribution in the correct direction?

**Benchmark.** StressTest (HUJI 2025) — 218 audio clips, same sentence × different stress placement, two candidate meanings, one correct.

**Today's demo covers:** dataset → extraction pipeline → pathway accuracies (primary / text-only / cascade) → decision-flip rate with 95% bootstrap CI.

Headline 2 (paired Wilcoxon vs noise floor, p ≈ 10⁻¹⁸) is done and lives in `results/analysis_summary.json`; it's mentioned at the end but not re-run here.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
PROJECT = '/content/drive/MyDrive/Colab_Notebooks/Independent_Study'
os.chdir(PROJECT)
sys.path.insert(0, PROJECT)

print('CWD:', os.getcwd())
!ls scripts
print('---')
!ls results | head -20

In [ ]:
!pip install -q datasets soundfile librosa scipy bitsandbytes

## 2. The dataset — what stress *sounds* like

Each sentence is recorded with two or three different stress placements. Different stressed word → different intended meaning. Let's open the dataset and play one clip so you can hear it.

In [ ]:
from datasets import load_dataset, Audio
import IPython.display as ipd

ds = load_dataset('slprl/StressTest', split='test')
ds = ds.cast_column('audio', Audio(sampling_rate=16000))
print(f'N clips: {len(ds)}   columns: {ds.column_names}')

s = ds[0]
print('\nTranscription :', s['transcription'])
print('Stressed word :', s['stress_pattern']['words'])
print('Choices       :', s['possible_answers'])
print('Gold label    :', s['label'], '→', s['possible_answers'][s['label']])
ipd.Audio(s['audio']['array'], rate=16000)

### Dataset arithmetic

```
85 sentences × 2 stress versions = 170 clips
16 sentences × 3 stress versions =  48 clips
                                   ────────────
                                    218 clips
```

Pairing each version with its sisters gives **133 within-sentence pairs**. Filtering to pairs where the two sister clips have *different* gold answers leaves **74 eligible pairs** for the decision-flip metric.

## 3. Verify token IDs

The Qwen tokenizer distinguishes `' A'` (with leading space) from `'A'` — different IDs. We hard-code the right ones after empirical verification, so the pipeline never reads the wrong column of the logit vector.

In [ ]:
!python scripts/verify_tokens.py

### Aside — the four possible answer tokens, and why we only read two

When the Format A prompt ends with `Answer:` (no trailing space), the model's next token could in principle be any of four candidates:

| Token | ID | What it means |
|---|---|---|
| `' A'` | 362 | space + A — the natural continuation after a colon |
| `' B'` | 425 | space + B — same |
| `'A'`  | 32  | bare A — would mean "no space," unusual after a colon |
| `'B'`  | 33  | bare B — same |

A well-trained LM almost always emits the space-prefixed version after a colon, so the pipeline reads **just two logits** — at positions **362** and **425** — and ignores 32 / 33.

**Why this is safe.** The Strategy 1 ↔ Strategy 2 smoke test (next cell) checks exactly this. Strategy 1 calls `model.generate(...)` and asks "what token did you actually emit?" If the answer were ever bare `'A'` or `'B'`, the smoke test's strict-match logic would fail. It passed on all 20 items — confirming the model concentrates probability on `' A'` / `' B'`, and the two-logit shortcut is sound.

**Format B handles the space differently.** The prompt is built to end with `"Answer: "` (trailing space already there), so the next token is just `'1'` (16) or `'2'` (17), no leading space needed. The candidates `' 1'` / `' 2'` don't exist as single tokens on this tokenizer (you saw that in the `verify_tokens.py` output — `[220, 16]` and `[220, 17]`), so there's no ambiguity to worry about.

**The "FAILED" line at the end of `verify_tokens.py` is expected.** It flags `' 1'` / `' 2'` as multi-token, but those are exactly the strings we *don't* use in the pipeline. The four token IDs we actually rely on (362, 425, 16, 17) all pass round-trip decoding cleanly.

## 4. The two prompt formats

We run the same experiment twice with two different answer-token choices, as a cheap robustness check.

In [ ]:
ans = s['possible_answers']
prompt_A = (
    "Out of the following answers, according to the speaker's stressed words, "
    "what is most likely the underlying intention of the speaker?\n"
    f"A. {ans[0]}\nB. {ans[1]}\nAnswer:"
)
prompt_B = s['audio_lm_prompt'] + ' '   # trailing space — Format B
print('=== Format A ===\n' + prompt_A)
print('\n=== Format B ===\n' + prompt_B)
print("\nFormat A → predicts token ' A' (362) or ' B' (425)")
print("Format B → predicts token '1' (16) or '2' (17)")

## 5. Extraction strategy

**Strategy 2.** One `model.forward()` pass. Read the logits at the final prompt position, then take the 2-way softmax over the two answer-token logits only. Cheaper than `generate()` and gives us the probability *magnitude*, not just the winner.

```
P('1' | audio, prompt)  =  exp(L₁) / (exp(L₁) + exp(L₂))
P('2' | audio, prompt)  =  exp(L₂) / (exp(L₁) + exp(L₂))
```

### Smoke test (Strategy 1 ↔ Strategy 2 agreement, N=20)

Confirms that the argmax we'd get from a full `generate()` call matches the argmax we read from the `forward()` logits. If these disagree, everything downstream is silently broken.

In [ ]:
!python scripts/smoke_test.py

### Live mini-extraction (5 items)

Proves the full extraction script runs end-to-end. The full 218-item run was done earlier — re-running it now would take ~15 min on a T4.

In [ ]:
!python scripts/extract_logits.py --format A --start 0 --end 5 \
    --out-dir results/_demo
print('---')
!head -2 results/_demo/logits_primary_A.jsonl

## 6. Full pre-computed results

From here on, we load the JSONL files produced by the full 218-item runs of the three pathways:

- **Primary** — Qwen2-Audio listens to the waveform directly
- **Text-only** — Qwen2-7B reads the gold transcript (no audio)
- **Cascade** — Whisper transcribes the audio, Qwen2-7B answers from the transcript

In [ ]:
import json
import numpy as np
from collections import defaultdict

def load(path):
    return [json.loads(l) for l in open(path)]

primary_A = load('results/logits_primary_A.jsonl')
primary_B = load('results/logits_primary_B.jsonl')
text_A    = load('results/logits_textonly_A.jsonl')
text_B    = load('results/logits_textonly_B.jsonl')
cascade_A = load('results/logits_cascade_A.jsonl')
cascade_B = load('results/logits_cascade_B.jsonl')

print('Loaded:', {
    'primary_A': len(primary_A), 'primary_B': len(primary_B),
    'text_A':    len(text_A),    'text_B':    len(text_B),
    'cascade_A': len(cascade_A), 'cascade_B': len(cascade_B),
})

## 7. Argmax accuracy per pathway

In [ ]:
import pandas as pd

def acc(rows):
    return np.mean([r['correct'] for r in rows])

rows = []
for name, A, B in [
    ('Primary (audio)', primary_A, primary_B),
    ('Text-only',       text_A,    text_B),
    ('Cascade',         cascade_A, cascade_B),
]:
    rows.append({
        'Pathway':      name,
        'Format A acc': f'{acc(A)*100:.1f}%',
        'Format B acc': f'{acc(B)*100:.1f}%',
    })
pd.DataFrame(rows)

**Read this.** Argmax accuracy hovers near 50% across the board — the StressTest paper sees the same thing. At the "final answer" level no pathway looks impressive. Importantly, text-only and cascade are at chance, which means the *words alone* don't leak the answer — any signal the primary audio model has must come from the prosody. Falsification controls clean.

Sub-decision shifts (next section, plus Wilcoxon below) are where the actual story lives.

### Why pair-consistency is the better metric than single-item accuracy

**Single-item accuracy** judges every clip independently: of the 218 clips, how many did the model get right? Each clip is its own coin flip. The problem: a model that just *guesses* "answer 1" every time scores ~50% — without any understanding of stress.

**Pair-consistency** uses the dataset's structure. The 218 clips aren't independent — they're recorded as *sister pairs*: same sentence, different stress, different correct answer. For each of the 133 sister pairs, ask: did the model get **both** clips of the pair right?

Getting both right is only possible if the model:
- correctly identified meaning 1 when stress favored meaning 1, **and**
- correctly identified meaning 2 when stress favored meaning 2.

That requires actually tracking the stress between the two clips.

#### Chance × chance — what a random model would score

- A coin-flip model: P(both right) = 0.5 × 0.5 = **25%**.
- A "lazy" model that always guesses the same answer: scores **0%** — because the two clips of a pair have opposite gold answers by construction.

The text-only and cascade pathways land at **8.3%** — that's the practical "no-stress baseline" for this dataset.

#### Why this metric is stricter

Single-item accuracy gives partial credit for half-understanding. Pair-consistency doesn't — you have to nail *both* members of the pair. The only way to do that consistently is by reading the prosodic difference between them.

## 8. Pair-consistency accuracy

A stricter test than single-item accuracy: of all sister pairs in the dataset, on how many does the model get **both** versions right? A model that's truly using stress to disambiguate should do this much better than chance × chance.

### Wait — 8.3% is *below* chance (25%). What does that mean?

Good catch. The naive "25% chance" assumes the two clips of a pair are judged independently. They aren't. Here's what's actually going on.

**Text-only and cascade see *identical* inputs on sister clips.**

For both clips of a sister pair:
- Text-only gets the same gold transcript twice (e.g. *"I never said he stole it"* for both)
- Cascade gets the same Whisper transcript twice (Whisper writes down words, not stress)

A deterministic forward pass on identical inputs gives identical outputs. The model is *forced* to give the same answer to both clips. It literally cannot tell them apart.

**Why that drives pair-both-correct way below 25%.**

Split the 133 pairs into two groups:

| Group | Count | What happens for text/cascade |
|---|---|---|
| Sisters with *different* gold labels | 74 | Same answer to both → exactly one is right, one is wrong → **0% pair-both-correct on this group, guaranteed** |
| Sisters with *same* gold label | 59 | Same answer to both → either both right or both wrong |

The 74 different-label pairs are pre-lost — no amount of skill helps. The 8.3% (≈ 11 pairs out of 133) all come from the 59 same-label group, where the model happened to land on the correct shared answer.

**The implication**

1. **The dataset is adversarial by design.** It deliberately puts the answer in the prosody for most sister pairs. Any model without ears is structurally guaranteed to fail on >55% of the pairs.
2. **Text-only / cascade scoring 8.3% means the falsification controls are doing exactly what they should** — they're being punished precisely at the spots where stress is the only cue.
3. **8.3% is the ceiling for any prosody-blind model on this metric, not a floor.** The only way to break it is to actually distinguish sister clips, which requires hearing the stress.

That's why primary's 13.5% / 19.5% matters so much. It isn't "a bit better than chance" — it's the only number on the table that broke the prosody-blind ceiling. The lift comes entirely from clips the text pathways were structurally unable to win.

In [ ]:
def group_by_sentence(rows):
    g = defaultdict(list)
    for r in rows:
        key = r.get('transcription') or r.get('gold_transcription')
        g[key].append(r)
    return g

def pair_both_correct(rows):
    g = group_by_sentence(rows)
    n_pairs, n_both = 0, 0
    for items in g.values():
        for i in range(len(items)):
            for j in range(i + 1, len(items)):
                n_pairs += 1
                if items[i]['correct'] and items[j]['correct']:
                    n_both += 1
    return n_both / n_pairs, n_pairs

for name, A, B in [
    ('Primary', primary_A, primary_B),
    ('Text',    text_A,    text_B),
    ('Cascade', cascade_A, cascade_B),
]:
    pcA, n = pair_both_correct(A)
    pcB, _ = pair_both_correct(B)
    print(f'{name:8s}  A={pcA*100:5.1f}%   B={pcB*100:5.1f}%   (N={n} pairs)')

**Read this.** Text-only and cascade get both sister items right only ~8% of the time. Primary audio doubles that (13–20%). The argmax accuracies looked similar — but the pairwise metric reveals that primary is the only pathway actually tracking stress across the two versions of the same sentence.

### What if 0% had been *inside* the CI? (the counterfactual)

Worth knowing precisely because the project's main claim doesn't depend on it.

#### The literal reading

A CI like `−2% to +12%` straddles zero. The point estimate (say 5%) might look positive, but the data are also consistent with no effect — or with a small effect in the *wrong* direction. We couldn't rule out that the observed positive number is just noise from having only 74 pairs.

#### What it would mean for the project's claims

- **Headline 1 (decision-flip rate) would lose its weight.** We couldn't write *"the model flips its argmax with the stress more often than against it."* We'd have to soften to: *"on these 74 pairs we observed a slight positive flip rate, but the 95% CI does not exclude zero."*

#### What would *not* be killed

- **Headline 2 (paired Wilcoxon, p ≈ 10⁻¹⁸)** still stands. It asks a different question:
  - Headline 1: does the **argmax flip** with stress?
  - Headline 2: does the **sub-decision logit distribution shift** with stress, even when the argmax doesn't flip?
  
  The whole *point* of the project is that the sub-decision answer can be positive while the argmax looks flat. A null Headline 1 alongside a strong Headline 2 would actually be the **most predicted** outcome under the project's hypothesis: *"the model hears prosody but isn't confident enough to flip its top-1 answer."*

- **Pair-consistency** (primary 13.5–19.5% vs text 8.3%) still stands. It uses all 133 pairs and a different baseline.

#### How the writeup would frame it

> *"At the argmax level the effect is too small to detect with 74 pairs (CI includes zero). But below the argmax, where the project's hypothesis actually lives, the effect is overwhelming (p ≈ 10⁻¹⁸). This is the exact picture predicted by 'the model hears stress but doesn't act on it confidently' — and it's why measuring sub-decision shifts mattered in the first place."*

A null at the argmax level would have *vindicated* the choice to look at sub-decision logits, not weakened it.

#### The actual situation

Both observed CIs (4.1–16.2% and 5.4–20.3%) sit fully above zero, so we get a small but real argmax-level bonus on top of the headline. The point of laying out this counterfactual: **even in the worst case the project's main claim survives**, because Headline 2 was always the load-bearing result.

## 9. Headline 1 — Decision-flip rate (with 95% bootstrap CI)

For every sister pair with **different gold answers** (74 pairs), check: when stress changes, does the model's argmax flip in the *correct* direction (each version landing on its own gold label)?

**Bootstrap CI** — resample the 74 indicators with replacement 10,000 times, recompute the flip rate each time, report the 2.5th and 97.5th percentiles. If zero falls outside that interval, the effect isn't sampling noise.

In [ ]:
def flip_indicators(rows):
    g = group_by_sentence(rows)
    inds = []
    for items in g.values():
        for i in range(len(items)):
            for j in range(i + 1, len(items)):
                a, b = items[i], items[j]
                if a['label'] == b['label']:
                    continue   # same-meaning sister pair — not eligible
                hit = int(a['argmax'] == a['label'] and b['argmax'] == b['label'])
                inds.append(hit)
    return np.array(inds, dtype=float)

def bootstrap_ci(arr, iters=10_000, alpha=0.05, seed=0):
    rng = np.random.default_rng(seed)
    n = len(arr)
    boots = np.empty(iters)
    for k in range(iters):
        boots[k] = rng.choice(arr, size=n, replace=True).mean()
    return np.percentile(boots, [100 * alpha / 2, 100 * (1 - alpha / 2)])

for fmt, rows in [('A', primary_A), ('B', primary_B)]:
    ind = flip_indicators(rows)
    rate = ind.mean()
    lo, hi = bootstrap_ci(ind)
    print(f'Format {fmt}: flip rate = {rate*100:5.2f}%   '
          f'95% CI = ({lo*100:.1f}%, {hi*100:.1f}%)   '
          f'N eligible pairs = {len(ind)}')

## 11. What's left — and why pair-consistency alone isn't the finish line

Pair-consistency proves the audio model is *somewhat* sensitive to stress at the **argmax** (final-answer) level. That's a real result, but it's a coarse one. Most of the project's scientific value lives in finer questions that pair-consistency can't answer.

### What pair-consistency does *not* address

- **How big the effect is.** 13.5–19.5% pair-both-correct means the model is still failing on 80–87% of pairs. Barely hearing? Or hearing strongly but unable to act?
- **Where the signal lives.** The model could be 49% confident in the right answer because of stress (a real prosodic shift) — pair-consistency would call that a failure, but the sub-decision logits would show the shift.
- **Whether the effect survives a strict statistical test.** Pair-consistency has no p-value, no noise-floor comparison.
- **Which component is responsible** — audio encoder ("ears") or language backbone ("brain")?
- **How the effect depends on where the stress falls** in the sentence (initial / medial / final).

### What each remaining piece contributes

**Headline 2 — paired Wilcoxon vs noise floor (Week 4, already done).** Looks *inside* the logit distribution, not just at the argmax. For each item, compares the stress-induced logit shift against the strictest meaning-preserving perturbation shift. Result: p ≈ 4 × 10⁻¹⁸ in both formats — the sub-decision shift is real, orders of magnitude stronger than acoustic jitter. **This is the project's actual headline finding.** Pair-consistency is supporting evidence for it. Numbers live in `results/analysis_summary.json`.

**Text-rescue probe (Week 3, pending).** Capitalize the stressed word in the gold transcript (e.g. *"I NEVER said he stole it"*) and feed it to the text-LLM. Two possible outcomes:
- If accuracy jumps → **"ears failed"** → the LM backbone can use stress when it arrives as text; the audio encoder is the bottleneck.
- If accuracy stays at the text-only baseline → **"brain failed"** → the LM can't use stress even when handed to it directly; the LM is the bottleneck.

Either outcome converts "the model is bad at this" into "this specific component needs fixing." Pair-consistency tells us *that* there's a weakness; text-rescue tells us *where* it is.

**Figures + stress-position breakdown (Week 5).** Decompose the single decision-flip number by where the stress falls in the sentence. If final-stress flips reliably and initial-stress doesn't, that's a mechanistic finding about how the audio model integrates information over time — a separate claim from "stress matters." Figures also turn the Wilcoxon p-value into something a reader can see at a glance.

**Writeup + repo cleanup (Week 6).** Promote `docs/results.md` into a paper-style document. Make the scripts reproducible by someone other than us. The difference between a finding that exists and a finding that gets cited.

### One-line summary

> Pair-consistency proves the audio model is *somewhat* sensitive to stress at the argmax level. The remaining work measures **how strong** (Wilcoxon — done, p ≈ 10⁻¹⁸), **where the bottleneck is** (text-rescue — next), **how it varies with stress position** (Week 5), and **packages it for reuse** (Week 6). The argmax-level result is a hint; the sub-decision and architectural results are what make this a contribution.

## 10. Visualization — bootstrap distribution

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, (fmt, rows) in zip(axes, [('A', primary_A), ('B', primary_B)]):
    ind = flip_indicators(rows)
    rng = np.random.default_rng(0)
    boots = np.array([rng.choice(ind, size=len(ind), replace=True).mean()
                      for _ in range(10_000)])
    lo, hi = np.percentile(boots, [2.5, 97.5])
    ax.hist(boots * 100, bins=40, color='#6b4ca8', alpha=0.75)
    ax.axvline(0, color='black', linewidth=1, label='no-effect (0%)')
    ax.axvline(lo * 100, color='red', linestyle='--', label=f'95% CI [{lo*100:.1f},{hi*100:.1f}]')
    ax.axvline(hi * 100, color='red', linestyle='--')
    ax.axvline(ind.mean() * 100, color='green', linewidth=2, label=f'point est. {ind.mean()*100:.1f}%')
    ax.set_title(f'Format {fmt}: bootstrap distribution of decision-flip rate')
    ax.set_xlabel('flip rate (%)')
    ax.legend(loc='upper right', fontsize=8)
axes[0].set_ylabel('# bootstrap samples')
plt.tight_layout()
plt.show()

## 11. What's next

- **Headline 2 — paired Wilcoxon (already done).** For every item, compare the *stress-induced* logit shift against the *strictest noise-floor* shift (gain / additive noise / time-stretch). Result: p ≈ 4 × 10⁻¹⁸ (Format A), 8 × 10⁻¹⁷ (Format B). Stored in `results/analysis_summary.json` — produced by `scripts/analysis.py`. We won't re-run it here; one demo of bootstrap is enough.
- **Pre-registered threshold.** Failed because Qwen2-Audio is unusually fragile to ±2% time-stretching — a secondary model-robustness finding, documented honestly in the writeup.
- **Week 5–6.** Text-rescue probe (ears vs brain attribution), figures by stress position, paper-style writeup.